In [ ]:
!pip install -U transformers

## Local Inference on GPU
Model page: https://huggingface.co/lmqg/bart-large-squad-qg

⚠️ If the generated code snippets do not work, please open an issue on either the [model repo](https://huggingface.co/lmqg/bart-large-squad-qg)
			and/or on [huggingface.js](https://github.com/huggingface/huggingface.js/blob/main/packages/tasks/src/model-libraries-snippets.ts) 🙏

In [ ]:
# Use a pipeline as a high-level helper
from transformers import pipeline

pipe = pipeline("text-generation", model="lmqg/bart-large-squad-qg")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/317 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.decoder.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
BartForCausalLM LOAD REPORT from: lmqg/bart-large-squad-qg
Key                                                       | Status     |  | 
----------------------------------------------------------+------------+--+-
model.encoder.layers.{0...11}.self_attn.q_proj.weight     | UNEXPECTED |  | 
model.encoder.layers.{0...11}.fc1.weight                  | UNEXPECTED |  | 
model.encoder.layers.{0...11}.fc1.bias                    | UNEXPECTED |  | 
model.encoder.layers.{0...11}.self_attn.k_proj.bias       | UNEXPECTED |  | 
model.encoder.layers.{0...11}.final_layer_norm.weight     | UNEXPECTED |  | 
model.encoder.layers.{0...11}.self_attn.v_proj.bias       | UNEXPECTED |  | 
model.encoder.layers.{0...11}.self_attn.out_proj.bia

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

In [ ]:
# Load model directly
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("lmqg/bart-large-squad-qg")
model = AutoModelForSeq2SeqLM.from_pretrained("lmqg/bart-large-squad-qg")

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [ ]:
import torch
print(torch.cuda.get_device_name(0))
print(torch.cuda.get_device_properties(0))

AssertionError: Torch not compiled with CUDA enabled

In [ ]:
import torch
import nltk
from datasets import Dataset
from transformers import (
    BartForConditionalGeneration,
    BartTokenizer,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback
)

# 1. Hardware Detection (Local)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# 2. Tokenizer & Model
model_name = 'facebook/bart-base'
tokenizer = BartTokenizer.from_pretrained(model_name)
model = BartForConditionalGeneration.from_pretrained(model_name).to(device)

# 3. CPU-Friendly Training Arguments
training_args = Seq2SeqTrainingArguments(
    output_dir="./local_bart_results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=1e-5,
    per_device_train_batch_size=2,    # Smaller batch size for local memory
    per_device_eval_batch_size=2,
    num_train_epochs=10,              # Fewer epochs for CPU speed
    predict_with_generate=True,
    fp16=torch.cuda.is_available(),   # Only use fp16 if GPU is available
    load_best_model_at_end=True,
    report_to="none",
    remove_unused_columns=False
)

# Proceed with your 'tokenized_data' and 'trainer' setup as before...
print("Ready for local execution.")

Using device: cpu


vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/558M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/259 [00:00<?, ?it/s]

Ready for local execution.


In [ ]:
import json
from pathlib import Path

def load_all_papers(level):
    """Load all papers and extract Q&A pairs with level-specific prefixes"""
    all_qa_pairs = []

    # Prefix mapping to help the model distinguish tasks
    prefix = "[HIGHER] " if level == "higher" else "[ORDINARY] "

    if level == "higher":
        years = range(15, 26)
    else:
        years = [14, 15, 16, 17, 18, 19, 20, 21, 23, 24, 25]

    for paper_num in years:
        # MODIFIED: Use absolute path for filepath to ensure files are found
        filepath = f'/content/{level}_merged_20{paper_num:02d}_.json'
        if not Path(filepath).exists():
            print(f"File not found: {filepath}") # Added for debugging
            continue

        try:
            with open(filepath, 'r', encoding='utf-8') as f:
                paper_data = json.load(f)

            if isinstance(paper_data, dict):
                paper_data = [paper_data]

            for question_group in paper_data:
                if question_group.get('skip') is True:
                    continue

                context = question_group.get('context', '')

                for part in question_group.get('parts', []):
                    if part.get('skip') == True:
                        continue

                    question_text = part.get('text', '')
                    solutions = part.get('solution')

                    if not solutions or not question_text:
                        continue

                    answer_text = '; '.join([str(s).strip() for s in solutions if s])

                    # THE KEY CHANGE: Prepend the prefix to the question
                    # Format: "[HIGHER] Context text Question text"
                    combined_question = f"{prefix}{context} {question_text}".strip()

                    qa_pair = {
                        'level': level,
                        'question': combined_question,
                        'answer': answer_text
                    }
                    all_qa_pairs.append(qa_pair)

        except Exception as e:
            print(f"Error loading {level} paper {paper_num}: {e}")

    return all_qa_pairs

# Combine both datasets
higher_data = load_all_papers("higher")
ordinary_data = load_all_papers("ordinary")

# Final combined list for the Dataset.from_dict() step
all_training_data = higher_data + ordinary_data

print(f"Combined Dataset: {len(all_training_data)} total samples.")

Combined Dataset: 344 total samples.


In [ ]:
!pip install evaluate rouge_score
from transformers import EarlyStoppingCallback, Seq2SeqTrainer, Seq2SeqTrainingArguments
import evaluate
import numpy as np

# Load the ROUGE metric using the 'evaluate' library
rouge = evaluate.load("rouge")

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]

    # Decode predicted numbers back into text
    # Note: ensure 'tokenizer' is defined in your namespace
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)

    # Replace -100 in the labels (we can't decode -100)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # Calculate ROUGE score
    result = rouge.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True)
    return {k: round(v * 100, 4) for k, v in result.items()}

In [ ]:
import torch
from datasets import Dataset
from transformers import (
    BartForConditionalGeneration,
    BartTokenizer,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback
)


# --- 1. PREPARE THE DATA ---
# (Assuming 'all_training_data' is the list we created in the previous step)
# We convert our list of dictionaries into a Hugging Face Dataset object
raw_dataset = Dataset.from_dict({
    'question': [item['question'] for item in all_training_data],
    'answer': [item['answer'] for item in all_training_data]
})

# Split: 90% for learning, 10% for testing the student's progress
split_data = raw_dataset.train_test_split(test_size=0.1)

# --- 2. THE TOKENIZER ---
model_name = 'facebook/bart-base'
tokenizer = BartTokenizer.from_pretrained(model_name)

def preprocess_function(examples):
    # Prepare the inputs (Questions)
    model_inputs = tokenizer(
        examples['question'],
        max_length=512,
        truncation=True,
        padding='max_length'
    )

    # Prepare the targets (Answers)
    # We use 'with_target' logic to create 'labels'
    labels = tokenizer(
        text_target=examples['answer'],
        max_length=128,
        truncation=True,
        padding='max_length'
    )

    # CRITICAL: Tell the model to ignore padding tokens when calculating loss
    # We replace the pad_token_id with -100
    labels["input_ids"] = [
        [(l if l != tokenizer.pad_token_id else -100) for l in label]
        for label in labels["input_ids"]
    ]

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# Run the tokenizer on our data
tokenized_data = split_data.map(preprocess_function, batched=True)

# IMPORTANT: Remove original text columns as they interfere with DataCollator
tokenized_data = tokenized_data.remove_columns(["question", "answer"])

# --- 3. THE MODEL ---
model = BartForConditionalGeneration.from_pretrained(model_name)

# --- 4. TRAINING ARGUMENTS (The "Study Plan") ---
training_args = Seq2SeqTrainingArguments(
    output_dir="./bart_results",
    eval_strategy="epoch",      # Check progress after every epoch
    save_strategy="epoch",            # Must match evaluation_strategy for EarlyStopping
    learning_rate=1e-5,               # How fast the student learns
    per_device_train_batch_size=4,    # How many questions the student sees at once
    num_train_epochs = 20,
    predict_with_generate=True,
    fp16=True,
    load_best_model_at_end=True,      # Required for EarlyStopping to work
    metric_for_best_model="eval_loss",# Look at loss to decide when to stop
    greater_is_better=False,          # Lower loss is better# Makes training faster on Google Colab (T4 GPU)
    report_to="none",
    remove_unused_columns=False       # This is kept, but filtering above is more direct for the collator
)

# Data Collator: Handles "batching" the data neatly for the GPU
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

# --- 5. THE TRAINER (The "Teacher") ---
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_data["train"],
    eval_dataset=tokenized_data["test"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    # The 'patience' of 3 means if the model doesn't improve for 3 epochs, STOP.
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

# START TRAINING
print("Starting training..")
trainer.train()

# --- 6. SAVE YOUR WORK ---
model.save_pretrained("./final_math_model")
tokenizer.save_pretrained("./final_math_model")
print("Model saved successfully!")

Map:   0%|          | 0/309 [00:00<?, ? examples/s]

Map:   0%|          | 0/35 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/259 [00:00<?, ?it/s]

Starting training..


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

1. The "Validation Loss" TrendIf you look at your Validation Loss, notice how it behaved:It went down until Epoch 4 ($2.4299$).After Epoch 4, it started going up slowly ($2.46 \rightarrow 2.49$).What this means: This is a classic case of Overfitting.Imagine a student who stops trying to understand the logic of math and starts just memorizing the specific questions in the textbook. By Epoch 10, the model became "too focused" on your specific 300 questions and lost its ability to generalize to new, unseen questions.

2. The Loss Value (2.4)A loss of ~2.4 is a bit high for a BART model that has finished training. Usually, we want to see that number drop closer to 1.0 or lower.Reason: Your dataset is very small (around 344 pairs). For a model as big as BART, 344 examples is like trying to teach someone a whole language using only one page of a book.

In [ ]:
from transformers import BartForConditionalGeneration, BartTokenizer

# Load the model you just trained
path = "./final_math_model"
my_tokenizer = BartTokenizer.from_pretrained(path)
my_model = BartForConditionalGeneration.from_pretrained(path).to("cuda")

def test_model(question, level="higher"):
    # Add the prefix so the model knows the level
    prefix = "[HIGHER] " if level == "higher" else "[ORDINARY] "
    input_text = prefix + question

    # Encode and generate
    inputs = my_tokenizer(input_text, return_tensors="pt", truncation=True).to("cuda")
    summary_ids = my_model.generate(inputs["input_ids"], max_length=150, num_beams=4, early_stopping=True)

    return my_tokenizer.decode(summary_ids[0], skip_special_tokens=True)

# TEST IT
print("Higher Level Answer:", test_model("Define the term ecosystem", level="higher"))
print("Ordinary Level Answer:", test_model("Define the term ecosystem", level="ordinary"))

# Task
Improve the BART training pipeline for the Q&A dataset by implementing text augmentation to increase the training set size, increasing regularization (weight decay and dropout) to prevent overfitting on the small 344-sample dataset, and lowering the learning rate for more stable updates. Finally, evaluate the performance and compare the new validation loss trends against the previous baseline.

## Data Augmentation

### Subtask:
Implement simple text augmentation using synonym replacement to increase the size of the training dataset.


**Reasoning**:
I will import the necessary NLTK resources and implement a synonym replacement function to augment the existing training dataset as requested.



In [ ]:
import nltk
import random
from nltk.corpus import wordnet

# 1. Download necessary resources
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger')

def get_synonyms(word):
    """Find synonyms for a word using WordNet."""
    synonyms = set()
    for syn in wordnet.synsets(word):
        for l in syn.lemmas():
            synonym = l.name().replace('_', ' ')
            if synonym.lower() != word.lower():
                synonyms.add(synonym)
    return list(synonyms)

def synonym_replacement(text, n=2):
    """Replace n random non-stop words with synonyms."""
    words = nltk.word_tokenize(text)
    # Basic filtering: ignore very short words or numbers
    suitable_words = [w for w in words if len(w) > 3 and w.isalpha()]

    if not suitable_words:
        return text

    new_words = words.copy()
    random.shuffle(suitable_words)
    num_replaced = 0

    for random_word in suitable_words:
        syns = get_synonyms(random_word)
        if syns:
            synonym = random.choice(syns)
            # Find the index of the word in the original list and replace it
            for i, word in enumerate(new_words):
                if word == random_word:
                    new_words[i] = synonym
                    break
            num_replaced += 1
        if num_replaced >= n:
            break

    return " ".join(new_words)

# 3. Apply augmentation to all_training_data
augmented_samples = []
for item in all_training_data:
    new_question = synonym_replacement(item['question'], n=2)
    augmented_samples.append({
        'level': item['level'],
        'question': new_question,
        'answer': item['answer']
    })

# 4. Combine original and augmented data
augmented_training_data = all_training_data + augmented_samples

# 5. Print results
print(f"Original samples: {len(all_training_data)}")
print(f"Augmented samples: {len(augmented_samples)}")
print(f"Total dataset size: {len(augmented_training_data)}")

**Reasoning**:
The code failed because the 'punkt_tab' resource was missing from the NLTK downloads. I will add 'punkt_tab' to the download list and re-run the augmentation logic.



**Reasoning**:
The training set size has been successfully doubled through synonym augmentation. Now, I will update the preprocessing pipeline to use the 'augmented_training_data' and then configure the BART model with increased regularization (weight decay and dropout) and a lower learning rate to address the overfitting observed in the previous run.



In [ ]:
!pip install evaluate rouge_score
import nltk
import random
from nltk.corpus import wordnet
from datasets import Dataset
from transformers import (
    BartForConditionalGeneration,
    BartTokenizer,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback
)
import numpy as np
import evaluate

# --- 0. DATA AUGMENTATION (SYNONYMS) ---
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('punkt_tab')

def get_synonyms(word):
    synonyms = set()
    for syn in wordnet.synsets(word):
        for l in syn.lemmas():
            synonym = l.name().replace('_', ' ')
            if synonym.lower() != word.lower():
                synonyms.add(synonym)
    return list(synonyms)

def synonym_replacement(text, n=2):
    words = nltk.word_tokenize(text)
    suitable_words = [w for w in words if len(w) > 3 and w.isalpha()]
    if not suitable_words: return text
    new_words = words.copy()
    random.shuffle(suitable_words)
    num_replaced = 0
    for random_word in suitable_words:
        syns = get_synonyms(random_word)
        if syns:
            synonym = random.choice(syns)
            for i, word in enumerate(new_words):
                if word == random_word:
                    new_words[i] = synonym
                    break
            num_replaced += 1
        if num_replaced >= n: break
    return " ".join(new_words)

augmented_samples = []
for item in all_training_data:
    new_question = synonym_replacement(item['question'], n=2)
    augmented_samples.append({'level': item['level'], 'question': new_question, 'answer': item['answer']})

augmented_training_data = all_training_data + augmented_samples

# --- 1. PREPARE THE DATA ---
rouge = evaluate.load("rouge")

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple): preds = preds[0]
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    result = rouge.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True)
    return {k: round(v * 100, 4) for k, v in result.items()}

raw_dataset = Dataset.from_dict({
    'question': [item['question'] for item in augmented_training_data],
    'answer': [item['answer'] for item in augmented_training_data]
})
split_data = raw_dataset.train_test_split(test_size=0.1, seed=42)

# --- 2. TOKENIZATION ---
model_name = 'facebook/bart-base'
tokenizer = BartTokenizer.from_pretrained(model_name)

def preprocess_function(examples):
    model_inputs = tokenizer(examples['question'], max_length=512, truncation=True, padding='max_length')
    labels = tokenizer(text_target=examples['answer'], max_length=128, truncation=True, padding='max_length')
    labels["input_ids"] = [[(l if l != tokenizer.pad_token_id else -100) for l in label] for label in labels["input_ids"]]
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_data = split_data.map(preprocess_function, batched=True)
tokenized_data = tokenized_data.remove_columns(["question", "answer"])

# --- 3. MODEL & TRAINING ---
model = BartForConditionalGeneration.from_pretrained(model_name, dropout=0.2, attention_dropout=0.2)

training_args = Seq2SeqTrainingArguments(
    output_dir="./bart_improved_results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=5e-6,
    weight_decay=0.01,
    per_device_train_batch_size=4,
    num_train_epochs=30,
    predict_with_generate=True,
    fp16=True,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none",
    remove_unused_columns=False
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_data["train"],
    eval_dataset=tokenized_data["test"],
    data_collator=DataCollatorForSeq2Seq(tokenizer, model=model),
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=5)]
)

print("Starting consolidated improved training...")
trainer.train()
model.save_pretrained("./improved_math_model")
tokenizer.save_pretrained("./improved_math_model")
print("Improved model saved successfully!")

## Add punkt_tab to NLTK

### Subtask:
Update the NLTK initialization to ensure all required tokenization resources are available, specifically addressing the missing 'punkt_tab' dependency.


**Reasoning**:
I will update the NLTK resource downloads to include 'punkt_tab', which is necessary for the tokenizer to function correctly during data augmentation.



In [ ]:
import nltk

# Ensure all required NLTK resources are downloaded
resources = [
    'wordnet',
    'omw-1.4',
    'punkt',
    'punkt_tab',
    'averaged_perceptron_tagger'
]

for res in resources:
    nltk.download(res)

print("NLTK resources successfully updated.")

**Reasoning**:
The NLTK resources have been successfully updated, including the previously missing 'punkt_tab'. I will now implement the back-translation augmentation as requested to further diversify the training samples.



**Reasoning**:
The error `KeyError: 'translation'` occurs because the specific task name used in the `pipeline` call ('translation_en_to_fr') is failing to resolve correctly in the transformers library. I will fix this by using the more generic task name 'translation' and manually passing the model, and I will also ensure the `sacremoses` library is installed, which is a required dependency for the Helsinki-NLP models.



In [ ]:
!pip install sacremoses
from transformers import pipeline
import torch

# 1. Initialize translation pipelines with a more robust task name definition
device = 0 if torch.cuda.is_available() else -1

# Use 'translation' task and provide the specific model
en_fr_translator = pipeline("translation", model="Helsinki-NLP/opus-mt-en-fr", device=device)
fr_en_translator = pipeline("translation", model="Helsinki-NLP/opus-mt-fr-en", device=device)

def back_translate(text):
    """Translates English text to French and then back to English."""
    try:
        # Translate to French
        fr_result = en_fr_translator(text, max_length=512)
        fr_text = fr_result[0]['translation_text']
        # Translate back to English
        en_back_result = fr_en_translator(fr_text, max_length=512)
        en_back_text = en_back_result[0]['translation_text']
        return en_back_text
    except Exception as e:
        print(f"Error during back-translation: {e}")
        return text

# 2. Generate back-translated samples
print("Starting back-translation... this may take a few minutes.")
back_translated_samples = []
# Using a sample to check speed/errors first if needed, but here processing all_training_data
for item in all_training_data:
    new_question = back_translate(item['question'])
    back_translated_samples.append({
        'level': item['level'],
        'question': new_question,
        'answer': item['answer']
    })

# 3. Combine with original and synonym-augmented data
# augmented_training_data already contains original + synonym samples
final_augmented_data = augmented_training_data + back_translated_samples

print(f"Original samples: {len(all_training_data)}")
print(f"Total augmented dataset size: {len(final_augmented_data)}")
print("Back-translation augmentation complete.")